In [ ]:
import pandas as pd

df = pd.read_csv("data_to_learn.csv")

df.drop(columns=["Unnamed: 0"],inplace=True)

df

In [ ]:
senders = set(df["sender"].unique())
len(senders)

In [ ]:
zero_events_df = df.loc[df["events_count"]==0].drop(columns=["events_count"])
pd.pivot_table(zero_events_df,values="file_size_bytes",index="sender",aggfunc=["mean","max","min"])
ZERO_EVENTS_SIZE=64

In [ ]:
min_size_per_count = pd.pivot_table(df,values="file_size_bytes",index="sender",aggfunc="min",columns="events_count",fill_value=0)
mspt = min_size_per_count.transpose()

In [ ]:
aredu_mspt = mspt["koko1"]
aredu_mspt_nzv = aredu_mspt.loc[(aredu_mspt.values != 0)]


In [ ]:
event_count = aredu_mspt_nzv.index
size_in_bytes = aredu_mspt_nzv.to_list()

In [ ]:
import matplotlib.pyplot as plt

number_of_events = aredu_mspt_nzv.index.to_list()
size_of_file = aredu_mspt_nzv.to_list()
# plotting the points   
plt.plot(number_of_events,size_of_file)
plt.xlabel("Number of records per file")
plt.ylabel("The Size of the file")

In [ ]:

ratio_size_per_event = [(size_of_file1-ZERO_EVENTS_SIZE)/number_of_events1 for size_of_file1,number_of_events1 in zip(size_of_file,number_of_events)]
#print(ratio_size_per_event)
plt.plot(number_of_events,ratio_size_per_event,label="Size of event")
plt.hlines(y=sum(ratio_size_per_event)/len(ratio_size_per_event),xmin=0,xmax=4000,colors='r',label="Average Size of Event")
plt.vlines(x=1,ymin=0,ymax=101,colors='b')
plt.xlabel("number_of_events")
plt.ylabel("Size of Event")
plt.legend()
print("Average size of event: ",sum(ratio_size_per_event)/len(ratio_size_per_event))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
nrdf = pd.read_csv("estiated_counts.csv")
nrdf = nrdf.drop("Unnamed: 0",axis=1)
undecodable_nrt_df = nrdf.loc[nrdf["count_events_decoded"]<0].drop(["count_events_decoded","prefix","exist"],axis=1)
undecodable_nrt_df.to_csv("results/undecodable_nrt_df.csv")

In [ ]:
clear_nrdf = nrdf.loc[nrdf["count_events_decoded"]!=-1]
clear_nrdf["diff"] = clear_nrdf["e_count_event"]-clear_nrdf["count_events_decoded"]
clear_nrdf.sort_values("file_size",inplace=True)
plt.plot(clear_nrdf["file_size"],clear_nrdf["diff"],label="Size of event")

In [ ]:
plt.plot(clear_nrdf["e_count_event"],clear_nrdf["count_events_decoded"],label="Size of event")

In [ ]:
print("sum e_count_event:",clear_nrdf["e_count_event"].sum())
print("sum count_events_decoded:",clear_nrdf["count_events_decoded"].sum())
ratio = [a-b for a,b in zip(clear_nrdf["e_count_event"],clear_nrdf["count_events_decoded"])]
plt.plot( range(1,len(ratio)+1) ,ratio,label="Size of event")
print(24844/130000000)

In [ ]:
res = clear_nrdf.drop(["Unnamed: 0","file_path","exist","file_size","estimate_count_event","e_count_event","diff"],axis=1)
res["connection"] = res["file_name"].str[:12]
res["type"] = res.apply(lambda row: "value1" if row["prefix"] == "AB" else "value2",axis=1) 
res["direction"] = res.apply(lambda row: "OUT" if row["file_name"][2:7] == "98765" else "IN",axis=1) 
res["file_count"] = res.apply(lambda row: 1, axis=1)
res[["connection","type","direction","creation_time","file_count","count_events_decoded","file_name"]].to_csv("results/decodable_nrt_df.csv")